In [2]:
import pickle

# Path to your pickle file
pickle_file_path = '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/cell-graph-tokenizer/human_cohort1_2m/token_dictionary.pkl'

# Load the pickle file
with open(pickle_file_path, 'rb') as file:
    token_dict = pickle.load(file)

# Now `token_dict` is a Python dictionary
print(token_dict)

{'<pad>': 0, '<mask>': 1, '<cls_0>': 2, '<cls_1>': 3, '<cls_2>': 4, '<cls_3>': 5, '<cls_4>': 6, '<cls_5>': 7, '<cls_6>': 8, '<cls_7>': 9, '<cls_8>': 10, '<cls_9>': 11, '<cls_10>': 12, '<cls_11>': 13, '<cls_12>': 14, '<cls_13>': 15, '<cls_14>': 16, '<cls_15>': 17, '<cls_16>': 18, '<cls_17>': 19, '<cls_18>': 20, '<cls_19>': 21, '<cls_20>': 22, '<cls_21>': 23, '<cls_22>': 24, '<cls_23>': 25, '<cls_24>': 26, '<cls_25>': 27, '<cls_26>': 28, '<cls_27>': 29, '<cls_28>': 30, '<cls_29>': 31, '<cls_30>': 32, '<cls_31>': 33, '<cls_32>': 34, '<cls_33>': 35, '<cls_34>': 36, '<cls_35>': 37, '<cls_36>': 38, '<cls_37>': 39, '<cls_38>': 40, '<cls_39>': 41, '<cls_40>': 42, '<cls_41>': 43, '<cls_42>': 44, '<cls_43>': 45, '<cls_44>': 46, '<cls_45>': 47, '<cls_46>': 48, '<cls_47>': 49, '<cls_48>': 50, '<cls_49>': 51, '<cls_50>': 52, '<cls_51>': 53, '<cls_52>': 54, '<cls_53>': 55, '<cls_54>': 56, '<cls_55>': 57, '<cls_56>': 58, '<cls_57>': 59, '<cls_58>': 60, '<cls_59>': 61, '<cls_60>': 62, '<cls_61>': 63, 

In [4]:
spv_count = sum(1 for key in token_dict if "spv" in key)
spv_count

46

In [5]:
len(token_dict)

5419

In [ ]:
import os
import pickle

import scanpy as sc
import squidpy as sq

In [ ]:
folder_path = '/lustre/scratch126/cellgen/team361/DATASETS/silver/xhs1000-39b_1p'
token_dict_path = '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/cell-neighborhood-tokenizer/xhs1000-39b_1p/token_dictionary.pkl'

In [ ]:
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

In [ ]:
len(token_dict)

In [ ]:
count_spv_keys = sum(1 for key in token_dict if key.startswith("spv"))
count_spv_keys

In [ ]:
token_dict[list(token_dict.keys())[-1:][0]]

In [ ]:
token_dict['ENSG00000292373']

In [ ]:
max_nonzero_counts = 0  # Initialize a variable to store the maximum count
neighbor_max_nonzero_counts = 0

# Iterate through all .h5ad files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith('.h5ad'):
        file_path = os.path.join(folder_path, filename)
        
        # Load the .h5ad file
        adata = sc.read_h5ad(file_path)

        sq.gr.spatial_neighbors(adata,
                                coord_type='generic',
                                spatial_key='spatial',
                                delaunay=True,
                                set_diag=True)

        adata.layers['X_neighborhood'] = (
                adata.obsp['spatial_connectivities'].T @ adata.X)
        
        # Calculate nonzero counts per cell
        nonzero_counts = (adata.X > 0).sum(axis=1)
        neighbor_nonzero_counts = (adata.layers['X_neighborhood'] > 0).sum(axis=1)
        
        # Get the maximum nonzero count in the current file
        file_max_nonzero = nonzero_counts.max()
        file_max_neighbor_nonzero = neighbor_nonzero_counts.max()
        
        # Update the overall maximum if the current file has a higher value
        if file_max_nonzero > max_nonzero_counts:
            max_nonzero_counts = file_max_nonzero

        if file_max_neighbor_nonzero > neighbor_max_nonzero_counts:
            neighbor_max_nonzero_counts = file_max_neighbor_nonzero

In [ ]:
adata.obs

In [ ]:
max_nonzero_counts

In [ ]:
neighbor_max_nonzero_counts

In [ ]:
adata_test = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/bronze/xhs1000-1b_1p/adata_5k_xenium_skin.h5ad')

In [ ]:
adata_test.obs['info_id']

In [ ]:
batch_mapping_dict = {value: idx for idx, value in enumerate(adata_test.obs['info_id'].unique().tolist())}

In [ ]:
adata_test.obs

In [ ]:
batch_mapping_dict # 19, 11, 32